In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/generation_config.json
/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


!pip install -q -U kaggle-benchmarks

## Democratic Hammurabi Benchmark Task Definition

In [3]:
from hammurabi_env import DemocraticHammurabi

class LLMDemocraticHammurabi:
    """
    A wrapper around our updated DemocraticHammurabi environment for LLMs.
    """
    def __init__(self, max_years=12):
        self.env = DemocraticHammurabi(max_years=max_years)
        self.state = self.env.reset()
        self.last_info = self.env._build_info()

    def play_turn(self, action_land: float, action_civ_procure: float, action_caravan_trade: float, action_feed: float, action_plant: float, action_project: float):
        actions = [
            action_land,
            action_civ_procure,
            action_caravan_trade,
            action_feed,
            action_plant,
            action_project
        ]
        self.state, reward, done, info = self.env.step(actions)
        self.last_info = info
        return done, info.get('reason', '')

    def get_slm_payload(self):
        """Generates the rich text CLI-style report for the LLM."""
        env = self.env
        info = self.last_info
        year = int(env.year)
        pop = int(env.population)
        grain = int(env.grain)
        land = int(env.land)
        silver = int(env.silver)
        l_price = env.land_price
        g_price = env.grain_price

        f_appr = env.farmers_approval
        w_appr = env.workers_approval
        e_appr = env.elites_approval
        avg_appr = env.get_average_approval()
        yrs_to_election = 4 - (year % 4) if (year % 4) != 0 else 0
        farmer_pop = env.farmer_pop
        worker_pop = env.worker_pop
        elite_pop = env.elite_pop
        civ_pop = farmer_pop + elite_pop
        civ_grain = env.civilian_grain
        worker_food_need = worker_pop * 20
        civ_food_need = civ_pop * 20
        max_workable_land = farmer_pop * 10
        seed_need = min(land, max_workable_land)
        royal_annual_need = worker_food_need + seed_need
        total_annual_need = worker_food_need + civ_food_need + seed_need

        farmer_silver = env.farmer_silver
        elite_silver = env.elite_silver
        worker_silver = env.worker_silver
        civ_silver = farmer_silver + elite_silver + worker_silver
        total_m2 = silver + civ_silver
        f_avg = farmer_silver / max(1, farmer_pop)
        e_avg = elite_silver / max(1, elite_pop)
        w_avg = worker_silver / max(1, worker_pop)
        dom_price = env.domestic_grain_price
        civ_purchasing_power = int(civ_silver // max(0.1, dom_price))
        worker_payroll = worker_pop * 2.0

        civ_safe_reserve = (farmer_pop + elite_pop) * 40
        civ_excess = max(0, civ_grain - civ_safe_reserve)
        silo_sealed = getattr(env, "granary_sealed", False)

        report = []

        # --- PREVIOUS YEAR REVIEW (If year > 1) ---
        if year > 1:
            report.append(f">>> YEAR {year - 1} HARVEST & FINANCIAL REPORT:")
            
            yield_harvest = info.get("harvest_yield", 3)
            canal_bonus = info.get("canal_yield_bonus", 0)
            # Need to approximate acres planted based on total harvest if not stored
            h_total = info.get("harvest_total", 0)
            if h_total == 0:
                 # fallback calculation just for display consistency if harvest_total isn't perfectly exposed in all cases
                 h_total = info.get("civilian_harvest", 0) + info.get("king_tax", 0)
            k_tax = info.get("king_tax", int(h_total * 0.4))
            c_harv = info.get("civilian_harvest", h_total - k_tax)
            
            if canal_bonus > 0:
                report.append(f"  [+] Harvest Yield:        {yield_harvest} bushels/acre (Base: {yield_harvest - canal_bonus} + Canal Bonus: +{canal_bonus}) | Total Crop: {h_total:,} bu")
            else:
                report.append(f"  [+] Harvest Yield:        {yield_harvest} bushels/acre (Total Crop: {h_total:,} bu)")
            report.append(f"      - King's 40% Tax:     +{k_tax:,} bu -> Royal Silos")
            report.append(f"      - Civilian 60% Share: +{c_harv:,} bu -> Civilian Granary")

            active_proj = info.get("active_project", 0)
            proj_spent = info.get("project_spent", 0.0)
            reclaimed_land = info.get("reclaimed_land", 0)
            if active_proj > 0:
                report.append(f"  [+] State Infrastructure & Public Works:")
                if active_proj == 1:
                    report.append(f"      - Canal Dredging:     Cleared Euphrates silt for {int(proj_spent):,} silver -> +{canal_bonus} yield/acre across all fields!")
                elif active_proj == 2:
                    report.append(f"      - Granary Seal:       Fortified royal silos with bitumen for {int(proj_spent):,} silver -> 100% Rat-Proof this year!")
                elif active_proj == 3:
                    report.append(f"      - Land Reclamation:   Drained marshlands for {int(proj_spent):,} silver -> Reclaimed +{reclaimed_land:,} fertile acres!")
                stim_dividend = int(proj_spent * 0.20)
                if stim_dividend > 0:
                    report.append(f"      - Keynesian Dividend: Distributed {stim_dividend:,} silver directly to workers (+approval & private liquidity)!")

            report.append(f"  [+] Domestic Commerce & Private Capital:")
            civ_proc_grain = info.get("civ_procured_grain", 0)
            civ_proc_cost = info.get("civ_procured_cost", 0)
            caravan_sold = info.get("caravan_grain_sold", 0)
            caravan_earned = info.get("caravan_silver_earned", 0)
            caravan_bought = info.get("caravan_grain_bought", 0)
            caravan_spent = info.get("caravan_silver_spent", 0)
            w_paid = info.get("worker_wages_paid", 0.0)
            w_unpaid = info.get("worker_wages_unpaid", 0.0)
            w_spend = info.get("worker_market_spend", 0.0)
            f_grain_sold = info.get("farmer_grain_sold", 0)
            f_silver_earned = info.get("farmer_silver_earned", 0)
            e_profit = info.get("elite_profit", 0)
            civ_bought = info.get("civilian_grain_bought", 0)
            civ_silver_spent = info.get("civilian_silver_spent", 0)
            land_silver_tf = info.get("land_silver_transfer", 0)
            imm_silver = info.get("immigrant_silver_brought", 0.0)
            
            if civ_proc_grain > 0:
                report.append(f"      - Crown Procurement:  King bought {civ_proc_grain:,} surplus bu directly from Farmers for {civ_proc_cost:,} silver")
            if caravan_sold > 0:
                report.append(f"      - Foreign Caravan:    Exported {caravan_sold:,} bu to foreign merchants for +{caravan_earned:,} silver")
            if caravan_bought > 0:
                report.append(f"      - Foreign Caravan:    Imported {caravan_bought:,} bu foreign cargo from caravan for {caravan_spent:,} silver")
            if w_unpaid == 0:
                report.append(f"      - State Payroll:      Paid {int(w_paid):,} silver to workers")
            else:
                report.append(f"      - State Payroll:      [!] DEFAULT! Unpaid wages: {int(w_unpaid):,} silver (Workers unrest!)")
            if w_spend > 0:
                report.append(f"      - Elite Public Market: Workers spent {int(w_spend):,} silver at Elite market stalls")
            if f_grain_sold > 0:
                report.append(f"      - Agrarian Surplus:   Farmers sold {f_grain_sold:,} surplus bu to Elites for +{f_silver_earned:,} silver")
                ret_margin = info.get("retail_margin", int(f_silver_earned * 0.30))
                if ret_margin > 0:
                    report.append(f"      - Public Market Retail: Urbanites bought bread & beer (+{ret_margin:,} silver retail profit to Elites)")
            if e_profit > 0:
                report.append(f"      - Elite Dividends:    Patricians collected +{e_profit:,} silver commercial & land dividends")
            if civ_bought > 0:
                report.append(f"      - Emergency Bread:    Civilians bought {civ_bought:,} bu from Royal Silos for +{civ_silver_spent:,} silver")
            if land_silver_tf != 0:
                if land_silver_tf > 0:
                    report.append(f"      - Land Sale Revenue:  Treasury gained +{land_silver_tf:,} silver from selling land to Civilians")
                else:
                    report.append(f"      - Land Buy Outlay:    Treasury paid {abs(land_silver_tf):,} silver to Farmers & Elites for land")
            
            f_distress = info.get("farmer_debt_distress", False)
            e_flight = info.get("elite_capital_flight", False)
            if f_distress:
                report.append(f"      - [!] Farmer Debt Alert: Peasant savings depleted (< 20 shekels avg)! Debt distress")
            if e_flight:
                report.append(f"      - [!] Elite Capital Flight: Patrician liquidity low (< 300 shekels avg)!")

            rats = info.get("rats_ate", 0)
            if info.get("granary_sealed", False):
                report.append(f"  [+] Bitumen Seal Defense: Rats repelled by bitumen-sealed silos! 0 bushels lost!")
            elif rats > 0:
                report.append(f"  [!] RATS ATTACK:          Rats devoured {rats:,} bushels of grain from royal silos!")

            civ_rot = info.get("civilian_grain_rot", 0)
            if civ_rot > 0:
                report.append(f"  [!] Granary Weevils & Rot: {civ_rot:,} bu stale surplus rotted in civilian silos!")
            
            starved = info.get("workers_starved", 0) + info.get("civilians_starved", 0)
            if starved > 0:
                report.append(f"  [!] FAMINE DISASTER:      {starved:,} citizens starved to death!")
            else:
                report.append(f"  [+] All citizens were well fed.")
                
            report.append("")


        # --- CURRENT YEAR REPORT ---
        report.append(f"----------------------------------------------------------------------")
        report.append(f" [YEAR {year} OF 12]  |  Next Election in: {yrs_to_election} year(s)")
        report.append(f"----------------------------------------------------------------------")
        
        f_pct = (farmer_pop / max(1, pop)) * 100.0
        w_pct = (worker_pop / max(1, pop)) * 100.0
        e_pct = (elite_pop / max(1, pop)) * 100.0
        report.append(f"  Demographics:        {pop:,} citizens total")
        report.append(f"                       - Farmers:        {farmer_pop:,} ({f_pct:.1f}% | Max workable: {max_workable_land:,} acres)")
        report.append(f"                       - State Workers:  {worker_pop:,} ({w_pct:.1f}% | Fed & paid by King)")
        report.append(f"                       - Elites:         {elite_pop:,} ({e_pct:.1f}% | Landowners & Market Stalls)")
        
        seal_status = " [BITUMEN SEALED - 100% Rat-Proof]" if silo_sealed else " (Rats attack if > 5000)"
        report.append(f"  Grain Reserves:      - Royal Silos:    {grain:,} bushels{seal_status}")
        if civ_excess > 0:
            report.append(f"                       - Civilian Silos: {civ_grain:,} bushels (Safe 2-yr reserve: {civ_safe_reserve:,} bu | Rot risk: {civ_excess:,} bu)")
        else:
            report.append(f"                       - Civilian Silos: {civ_grain:,} bushels (Safe 2-yr reserve: {civ_safe_reserve:,} bu | 100% Safe)")
        report.append(f"                       - Total Kingdom:  {grain + civ_grain:,} bushels")
        
        farmer_food_need = farmer_pop * 20
        elite_food_need = elite_pop * 20
        est_market_works = min(int(max(0, civ_grain - farmer_food_need) * 0.50), (worker_pop + elite_pop) * 20)
        report.append(f"  Annual Grain Need:   - Total Kingdom:  {total_annual_need:,} bushels (Direct Food + Seed)")
        report.append(f"                       - Royal Need:     {royal_annual_need:,} bushels (Worker Food: {worker_food_need:,} + Seed: {seed_need:,})")
        report.append(f"                       - Farmers Food:   {farmer_food_need:,} bushels (20 bu/farmer self-fed from granary)")
        report.append(f"                       - Elite Estates:  {elite_food_need:,} bushels (20 bu/elite self-fed from granary)")
        if est_market_works > 0:
            report.append(f"                       - Market Works:   ~{est_market_works:,} bushels (Surplus raw grain milled/brewed by Elites)")
        report.append(f"                       - Seed for Crops: {seed_need:,} bushels (1 bu/acre)")
        
        report.append(f"  Monetary Vaults:     - Royal Vault:    {int(silver):,} shekels ({silver/max(1.0, total_m2)*100:.1f}% M2 | 100% Rat-Immune)")
        report.append(f"                       - Private Sector: {int(civ_silver):,} shekels ({civ_silver/max(1.0, total_m2)*100:.1f}% M2)")
        report.append(f"                         * Farmers Co-op: {int(farmer_silver):,} shekels ({f_avg:.1f} avg/person | Debt alert: <20)")
        report.append(f"                         * Elite Houses:  {int(elite_silver):,} shekels ({e_avg:.1f} avg/person | Flight alert: <300)")
        report.append(f"                         * State Workers: {int(worker_silver):,} shekels ({w_avg:.1f} avg/person)")
        report.append(f"                       - Total Currency: {int(total_m2):,} shekels circulating in Babylon")
        
        report.append(f"  State Payroll & Bread: Worker Wages:   {int(worker_payroll):,} shekels/yr (2/worker auto-paid from Vault)")
        report.append(f"                       - Civ Solvency:   {civ_purchasing_power:,} bu max bread peasants can buy from silos")
        
        report.append(f"  Land Owned:          {land:,} acres (Capacity: {max_workable_land:,} workable by {farmer_pop:,} farmers)")
        report.append(f"  Current Markets:     Land: {l_price:.1f} silver/acre | Caravan Grain: {g_price:.2f} silver/bu | Domestic Bread: {dom_price:.2f} silver/bu")
        report.append(f"  Faction Approval:    Farmers: {f_appr:.1f}% | Workers: {w_appr:.1f}% | Elites: {e_appr:.1f}%")
        report.append(f"  Average Approval:    {avg_appr:.1f}% (Required to win election: >= 45.0%)")
        report.append(f"----------------------------------------------------------------------")
        report.append("")

        # --- CALCULATE OPTIMAL / MAX VALUES FOR PROMPT ---
        max_land_buy = int(silver // l_price)
        civ_land_budget = farmer_silver + elite_silver
        max_civ_can_buy = int(civ_land_budget // l_price)
        max_land_sell = min(land, max_civ_can_buy)
        
        report.append(f"1. REAL ESTATE MARKET (Price: {l_price:.2f} silver/acre)")
        report.append(f"   Max you can buy with Vault Silver: {max_land_buy:,} acres (Owned: {land:,} acres)")
        report.append(f"   Max you can sell to Civilians:    {max_land_sell:,} acres (Civilians have {int(civ_land_budget):,} silver)")
        report.append(f"   [Note: Buying land injects capital to Farmers & Elites; Selling absorbs private silver]")
        report.append(f"   `action_land` (-1.0 to 1.0): e.g. 1.0 = buy max affordable, -1.0 = sell max possible")
        report.append("")

        max_dom_affordable = int(silver // max(0.1, dom_price))
        max_dom_buy = min(civ_excess, max_dom_affordable)
        
        report.append(f"2. DOMESTIC GRAIN PROCUREMENT (Peasant Wholesale: {dom_price:.2f} silver/bu)")
        if civ_excess > 0:
            report.append(f"   Available Peasant Surplus: {civ_excess:,} bu (Beyond safe 2-yr reserve | Rot risk!)")
            report.append(f"   Max you can buy from farmers: {max_dom_buy:,} bu (Total cost: {int(max_dom_buy * dom_price):,} silver)")
        else:
            report.append(f"   Civilian granaries have no surplus beyond safe 2-year reserve ({civ_safe_reserve:,} bu).")
        report.append(f"   `action_civ_procure` (0.0 to 1.0): Fraction of available civilian surplus to buy.")
        report.append("")

        caravan_silver = getattr(env, "caravan_silver", 15000.0)
        caravan_cargo = getattr(env, "caravan_cargo", 8000.0)
        max_caravan_can_buy = int(caravan_silver // g_price)
        max_export_possible = min(grain, max_caravan_can_buy)
        max_caravan_cargo_affordable = int(silver // g_price)
        max_import_possible = min(int(caravan_cargo), max_caravan_cargo_affordable)
        
        civ_deficit = max(0, civ_food_need - civ_grain)
        total_royal_burden = royal_annual_need + civ_deficit
        royal_diff = grain - total_royal_burden
        trade_advice = f"Surplus: +{royal_diff:,} bu safe to export" if royal_diff > 0 else (f"Deficit: -{abs(royal_diff):,} bu needed" if royal_diff < 0 else "Balanced")

        report.append(f"3. FOREIGN MERCHANT CARAVAN (Caravan Tariff: {g_price:.2f} silver/bu)")
        report.append(f"   Caravan Liquidity:       {int(caravan_silver):,} silver purse | {int(caravan_cargo):,} bu cargo for sale")
        report.append(f"   Direct Royal Silo Need:  {royal_annual_need:,} bu (Workers: {worker_food_need:,} + Seed: {seed_need:,})")
        report.append(f"   RECOMMENDED ACTION:      [{trade_advice}]")
        report.append(f"   Max you could export:    -{max_export_possible:,} bu")
        report.append(f"   Max you could import:    +{max_import_possible:,} bu")
        report.append(f"   `action_caravan_trade` (-1.0 to 1.0): e.g. 1.0 = import max affordable, -1.0 = export max possible")
        report.append("")

        report.append(f"4. FEEDING WORKERS")
        report.append(f"   State workers to feed: {worker_pop:,} | Food needed: {worker_food_need:,} bushels (20 bu/worker)")
        report.append(f"   [Civilian status: {civ_pop:,} farmers/elites have {civ_grain:,} bu reserve (need {civ_food_need:,} bu)]")
        report.append(f"   `action_feed` (0.0 to 1.0): Fraction of current royal grain used to feed state workers only.")
        report.append("")

        report.append(f"5. PLANTING CROPS")
        report.append(f"   1 bu/acre | Farmer capacity: {max_workable_land:,} acres")
        report.append(f"   `action_plant` (0.0 to 1.0): Fraction of remaining royal grain to plant as seed.")
        report.append("")

        report.append(f"6. STATE INFRASTRUCTURE & PUBLIC WORKS")
        report.append(f"   [0.26-0.50] Canal Dredging: 500 silver, >= 30 workers (boosts yield)")
        report.append(f"   [0.51-0.75] Granary Fortification: 400 silver, >= 20 workers (rat immune)")
        report.append(f"   [0.76-1.00] Land Reclamation: 600 silver, >= 35 workers (adds land)")
        report.append(f"   `action_project` (0.0 to 1.0): Float value mapping to the project choice.")
        
        return "\n".join(report)


You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, your budget, and the political polls.
2. Calculate your budget based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.
4. You must output your tool call using EXACTLY this syntax, replacing the values with your calculated numbers:
<|tool_call>call:issue_decree{acres_to_buy: [number], bushels_to_feed: [number], acres_to_plant: [number]}<tool_call|>

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve. Starvation drastically lowers Worker approval and causes instant impeachment if too high!
2. PLANTING: It costs exactly 1 bushe

In [4]:
import kaggle_evaluation.core.benchmarks as kbench

@kbench.task(name="democratic_hammurabi_v2")
def play_democratic_hammurabi(
    llm: kbench.Actor,
    max_years: int = 12,
    max_retries_per_turn: int = 3
) -> None:
    """
    Evaluates an LLM's ability to act as the Grand Vizier of Democratic Babylon,
    balancing resources, monetary policies, and political factions.
    """
    game = LLMDemocraticHammurabi(max_years=max_years)
    current_decree = []

    def issue_decree(
        action_land: float, 
        action_civ_procure: float, 
        action_caravan_trade: float, 
        action_feed: float, 
        action_plant: float, 
        action_project: float
    ) -> str:
        """
        Issues the royal decrees for the year, utilizing continuous floats.
        
        Args:
            action_land (float): -1.0 to 1.0 (negative to sell land, positive to buy land from civilians).
            action_civ_procure (float): 0.0 to 1.0 (fraction of affordable domestic civilian grain to buy for the royal silos).
            action_caravan_trade (float): -1.0 to 1.0 (negative to sell royal grain for silver to caravan, positive to buy foreign grain with royal silver).
            action_feed (float): 0.0 to 1.0 (fraction of royal grain to use to feed state workers only).
            action_plant (float): 0.0 to 1.0 (fraction of remaining royal grain to plant on land).
            action_project (float): 0.0 to 1.0 (0-0.25=None, 0.26-0.50=Canal Dredging, 0.51-0.75=Granary Fortification, 0.76-1.0=Land Reclamation).
        """
        
        # Clamp inputs just to be safe
        action_land = max(-1.0, min(1.0, float(action_land)))
        action_civ_procure = max(0.0, min(1.0, float(action_civ_procure)))
        action_caravan_trade = max(-1.0, min(1.0, float(action_caravan_trade)))
        action_feed = max(0.0, min(1.0, float(action_feed)))
        action_plant = max(0.0, min(1.0, float(action_plant)))
        action_project = max(0.0, min(1.0, float(action_project)))

        current_decree.clear()
        current_decree.extend([action_land, action_civ_procure, action_caravan_trade, action_feed, action_plant, action_project])
        return "The decrees have been recorded."

    system_prompt = """You are the Grand Vizier of Democratic Babylon.
I will provide you with a highly detailed 'REPORT' containing raw data about the kingdom's finances, demographics, and markets.
1. Think silently about the implications of the data and review the recommended limits for each action.
2. Determine your optimal strategy based on the calculations provided in the report.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.

STRATEGY GUIDE (KEYS TO WINNING):
- You ONLY need to feed the State Workers. Farmers and Elites are NOT provided food by the state; they feed themselves from their private granary.
- Your Royal Silos are funded by an automatic 40% grain tax on harvests.
- State workers are automatically paid 2 silver per year from the Vault.
- If civilian grain is oversupplied (beyond their 2-year safe reserve), they face rot. ALWAYS try to buy their surplus (`action_civ_procure`) and then export it for profit (`action_caravan_trade`).
- If you are in a drought or lack grain, fund the Canal Dredging project (`action_project` = 0.5) to boost yields, and import grain from the caravan.
- If your Royal Silos are heavily stocked (>5000 bu), rats will attack. Fund the Granary Bitumen Fortification project (`action_project` = 0.75) to make silos 100% rat-proof.
- In general, balancing Canal Dredging and Granary Fortification depending on your silo inventory is a highly successful strategy.

EXAMPLE THOUGHT PROCESS:
"The report shows civilian silos have 8,659 bu of rot risk surplus. I will buy all of it (action_civ_procure = 1.0) to save it and inject silver to peasants. My Royal Silos now have 22,283 bu total. That's a massive surplus, so I will export 10,000 bu to the caravan (action_caravan_trade = -1.0 max possible export) for huge silver profit. I have 52 workers, requiring 1040 bu to feed (action_feed = 1.0 of the required amount). I will plant max acres (action_plant = 1.0). Since I still have over 10,000 bu left in silos, rats will attack. I will fund Granary Bitumen Fortification (action_project = 0.75) to secure the food."

The report includes a 6-step breakdown of your options. You must issue your decree using the `issue_decree` tool, passing exactly 6 float values corresponding to those options to balance the factions and survive!"""
    
    kbench.system.send(system_prompt)
    
    done = False
    reason = ""
    
    while not done:
        report = "Current Kingdom Status:\n" + game.get_slm_payload()
        
        current_decree.clear()
        retries = 0
        prompt_text = report
        
        while len(current_decree) == 0:
            if retries >= max_retries_per_turn:
                done = True
                reason = "Impeached for analysis paralysis (failed to issue a valid decree after max retries)."
                break
                
            try:
                response = llm.prompt(prompt_text, tools=[issue_decree])
            except Exception as e:
                done = True
                reason = f"Impeached for analysis paralysis (LLM Error: {type(e).__name__})."
                break
            
            if len(current_decree) == 0:
                last_msg = kbench.chats.current().messages[-1]
                calls = getattr(last_msg, "tool_calls", [])
                
                if not calls and last_msg.sender != "tool":
                    prompt_text = "You did not issue a decree. You MUST call the `issue_decree` tool with 6 float values to end your turn."
                else:
                    prompt_text = "Please review the Accountant's error and recalculate your decree."
                retries += 1
            else:
                break
                
        if len(current_decree) == 6:
            done, reason = game.play_turn(*current_decree)
                
    survived = (game.env.year > max_years)
    final_score = game.env._calculate_reward()
    
    # Collect detailed final stats similar to the user's report
    stats_msg = (
        f"Game Over Reason: {reason or game.env.game_over_reason} | "
        f"Final Pop: {game.env.population} | "
        f"Final Royal Vault: {int(game.env.silver)} | "
        f"Final Private M2: {int(game.env.civilian_silver)} | "
        f"Final Royal Silos: {game.env.grain} | "
        f"Final Civ Granary: {game.env.civilian_grain} | "
        f"Final Land: {game.env.land} | "
        f"Final Score: {final_score:.1f}"
    )
    
    kbench.assertions.assert_true(
        survived,
        expectation=f"Agent must survive 12 years. | {stats_msg}"
    )


In [ ]:

# Run the benchmark
result = play_democratic_hammurabi.run(
    llm=kbench.llm,
    max_years=12
)

print("\n--- Benchmark Results ---")
print(f"Status: {result.status.name}")
if result.assertion_results:
    assertion = result.assertion_results[0]
    print(f"Passed: {assertion.passed}")
    print(f"Details: {assertion.expectation}")
